# Spoken Grammar Scoring — Final Evaluation Notebook
**Author: Priyansh Rathore**

## Objective
Predict a continuous grammar score between **0 and 5** from a spoken recording.
The supplied competition data contains **769 training recordings** and **216 test recordings**.
Test labels are placeholders and are never used for training, validation or model selection.

This notebook presents three clearly separated approaches:

| Approach | Features and model | Output file |
|---|---|---|
| **A. Audio + CatBoost** | Frozen WavLM-base-plus features, supporting speech statistics, CatBoost | `submission_audio_catboost.csv` |
| **B. Text/audio ensemble** | DeBERTa transcript predictions combined with Approach A; **no TF-IDF** | `submission_text_audio.csv` |
| **C. WavLM-large ensemble** | Frozen WavLM-large features, the same supporting statistics, 50:50 SVR/CatBoost | `submission_wavlm_large.csv` |

**Default final submission:** Approach C, also exported as `submission.csv`.
Only the components required for these approaches are included; no Qwen, TF-IDF or W2v-BERT stage is run.

### Important terminology
“Audio + CatBoost” is the experiment name used throughout development. Its predictors include
ASR confidence and a few transcript statistics in addition to audio embeddings. It is **not a pure
waveform-only ablation**. Keeping these features preserves the earlier experiment's design.

### How to run
1. Attach the competition dataset. In Kaggle Settings, choose **GPU T4 ×2** and enable **Internet**.
2. Start a fresh session and select **Run All**. No other notebook or manually created helper file is required.
3. Save an executed notebook version so the final evaluator can see actual metrics, figures and the report.
4. Submit `submission.csv` after reviewing the filename audit at the end.

The notebook assigns independent jobs to every detected GPU and shares available CPU cores between
workers. Only one model runs in each GPU process, and processes exit between stages.


## 1. Configuration and environment

All experiment choices are collected here. The speech models are frozen feature extractors;
only DeBERTa, CatBoost and SVR are fitted to the grammar labels.

- **Whisper:** beam size 3 for decoding; word timestamps disabled to avoid the alignment failure encountered previously.
- **Audio:** 16 kHz mono input, 8-second chunks, frame-weighted mean/std pooling.
- **Text:** DeBERTa-base, 256 tokens, microbatch 2, gradient accumulation, mixed precision and gradient checkpointing.
- **Disk:** download one weight format per model; keep small features/predictions; do not save optimizer or epoch checkpoints.

`REUSE_RUN` is optional. To reuse the earlier work, attach the previous Kaggle outputs and point it to
the original `shl_run_...` directory containing `data.csv`, `config.json`, `labels.npy` and `asr_*.json`.
If needed, set `REUSE_LARGE_RUN` to its `wavlm_large_...` subdirectory. Leave both as `None` for a full fresh run.
Never point these settings at a run from a different dataset.


In [ ]:
import os
import sys
import json
import hashlib
import shutil
import subprocess
from pathlib import Path

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
INPUT = Path("/kaggle/input")

REUSE_RUN = None          # Example: Path("/kaggle/input/old-outputs/shl_run_...")
REUSE_LARGE_RUN = None    # Usually found automatically beneath REUSE_RUN.
DATA_ROOT = None          # Set only if more than one competition dataset is attached.

SETTINGS = {
    "seed": 42,
    "folds": 5,
    "asr_beam_size": 3,
    "audio_chunk_seconds": 8,
    "text_epochs": 4,
    "text_max_length": 256,
    "text_batch_size": 2,
    "text_effective_batch": 16,
    "text_learning_rate": 2e-5,
}
MODEL_IDS = {
    "asr": "Systran/faster-whisper-large-v3",
    "audio_base": "microsoft/wavlm-base-plus",
    "audio_large": "microsoft/wavlm-large",
    "text": "microsoft/deberta-v3-base",
}

os.environ.update(
    HF_HOME=str(OUT / "shl_model_cache"),
    HF_HUB_DISABLE_PROGRESS_BARS="1",
    TOKENIZERS_PARALLELISM="false",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
)

def require_free_disk(gigabytes):
    free = shutil.disk_usage(OUT).free / 2**30
    if free < gigabytes:
        raise RuntimeError(
            f"Need {gigabytes:.1f} GB free; found {free:.1f} GB. "
            "Use a fresh session or remove only your own unneeded files."
        )

require_free_disk(9)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "--disable-pip-version-check", "transformers==4.57.1",
    "faster-whisper==1.2.1", "sentencepiece==0.2.1",
    "catboost>=1.2.8,<2", "soundfile>=0.12", "av<19",
    "psutil", "threadpoolctl",
], check=True)

import numpy as np
import pandas as pd
import torch
import psutil
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

CPU_CORES = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)
GPU_COUNT = torch.cuda.device_count()
print(f"CPU cores: {CPU_CORES} | GPUs: {GPU_COUNT}")
for gpu in range(GPU_COUNT):
    print(f"GPU {gpu}: {torch.cuda.get_device_name(gpu)}")
if GPU_COUNT == 0:
    print("CPU execution is supported, but speech processing and DeBERTa will be much slower.")


## 2. Read the data and resolve audio paths

Some training and test files share the same **filename**, but belong to different folders.
We resolve `train/filename` and `test/filename` separately. We reject cases where both splits
actually resolve to the same file path. Labels equal to zero are valid and are retained.


In [ ]:
if DATA_ROOT is None:
    candidates = [
        path.parent for path in INPUT.rglob("train.csv")
        if (path.parent / "test.csv").exists()
        and any(path.parent.rglob("*.wav"))
    ]
    assert len(candidates) == 1, f"Set DATA_ROOT explicitly. Candidates: {candidates}"
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT)

train = pd.read_csv(DATA_ROOT / "train.csv")
test = pd.read_csv(DATA_ROOT / "test.csv")
sample_path = DATA_ROOT / "sample_submission.csv"
sample = pd.read_csv(sample_path) if sample_path.exists() else None

assert {"filename", "label"}.issubset(train.columns)
assert "filename" in test.columns
for frame in (train, test):
    assert frame.filename.notna().all()
    frame["filename"] = frame.filename.astype(str)
    assert frame.filename.is_unique

y = pd.to_numeric(train.label, errors="raise").to_numpy(dtype=np.float32)
assert np.isfinite(y).all() and ((y >= 0) & (y <= 5)).all()
n_train, n_test = len(train), len(test)

audio_index = {}
for path in DATA_ROOT.rglob("*.wav"):
    audio_index.setdefault(path.name, []).append(path)

def resolve_audio(filename, split):
    direct = DATA_ROOT / split / filename
    if direct.is_file():
        return str(direct.resolve())
    matches = audio_index.get(Path(filename).name, [])
    preferred = [p for p in matches if split in p.relative_to(DATA_ROOT).parts[:-1]]
    matches = preferred or matches
    assert len(matches) == 1, f"Missing or ambiguous audio: {split}/{filename}"
    return str(matches[0].resolve())

data = pd.concat([
    train[["filename"]].assign(split="train"),
    test[["filename"]].assign(split="test"),
], ignore_index=True)
data["path"] = [resolve_audio(name, split) for name, split in zip(data.filename, data.split)]

train_paths = set(data.loc[data.split == "train", "path"])
test_paths = set(data.loc[data.split == "test", "path"])
assert train_paths.isdisjoint(test_paths), "Train/test resolve to the same audio paths."

print(f"Train: {n_train} | Test: {n_test} | Sample: {0 if sample is None else len(sample)}")
print("Shared filenames in separate folders:", len(set(train.filename) & set(test.filename)))
display(train.head())


## 3. Understand the recordings and labels

The distribution matters: the earlier dataset had 37 zero-score recordings, but only one recording
with score 1. Results for extremely rare labels are consequently uncertain. The duration plot also
checks that preprocessing should handle the observed lengths rather than assuming every file is exactly 60 seconds.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def inspect_audio(path):
    info = sf.info(path)
    return {"duration": info.frames / info.samplerate,
            "sample_rate": info.samplerate, "channels": info.channels}

with ThreadPoolExecutor(max_workers=CPU_CORES) as pool:
    audio_info = pd.DataFrame(list(pool.map(inspect_audio, data.path)))

display(train.label.value_counts().sort_index().rename("count").to_frame())
display(audio_info.iloc[:n_train].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y, bins=np.arange(-0.25, 5.51, 0.5), edgecolor="white")
axes[0].set(title="Training grammar scores", xlabel="Score", ylabel="Recordings")
axes[1].hist(audio_info.duration, bins=20, edgecolor="white")
axes[1].set(title="All recording durations", xlabel="Seconds", ylabel="Recordings")
plt.tight_layout()
plt.savefig(OUT / "data_overview.png", dpi=150)
plt.show()


## 4. Download models and establish a reproducible run

Each model is downloaded once and shared by the worker processes. We keep only one weight format
and record the exact model revision and installed package versions. A run directory separates
different configurations and input manifests, preventing accidental mixing of cached experiments.


In [ ]:
from huggingface_hub import HfApi, snapshot_download
from importlib.metadata import version

model_paths, model_revisions = {}, {}
api = HfApi()
for stage, model_id in MODEL_IDS.items():
    info = api.model_info(model_id, files_metadata=True)
    names = {item.rfilename for item in info.siblings}
    if stage == "asr":
        weight_name = "model.bin"
    else:
        weight_name = "model.safetensors" if "model.safetensors" in names else "pytorch_model.bin"
    assert weight_name in names, f"Unexpected checkpoint layout: {model_id}"
    selected = [item for item in info.siblings if "/" not in item.rfilename
                and (item.rfilename == weight_name or item.rfilename.endswith((".json", ".model", ".txt")))]
    require_free_disk(1.5 + sum(item.size or 0 for item in selected) / 2**30)
    model_paths[stage] = snapshot_download(
        model_id, revision=info.sha,
        allow_patterns=[item.rfilename for item in selected], max_workers=2,
    )
    model_revisions[stage] = info.sha

manifest = {
    "pipeline_version": 1,
    "settings": SETTINGS,
    "model_revisions": model_revisions,
    "train_csv": train[["filename", "label"]].to_csv(index=False),
    "records": [(row.split, row.filename, Path(row.path).stat().st_size,
                 Path(row.path).stat().st_mtime_ns) for row in data.itertuples()],
    "reuse_run": str(REUSE_RUN),
    "reuse_large_run": str(REUSE_LARGE_RUN),
}
run_key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:12]
RUN = OUT / f"shl_final_{run_key}"
for stage in ("asr", "audio_base", "audio_large", "text", "regressors", "logs"):
    (RUN / stage).mkdir(parents=True, exist_ok=True)

config = {**SETTINGS, "models": model_paths}
(RUN / "config.json").write_text(json.dumps(config, indent=2))
(RUN / "manifest.json").write_text(json.dumps(manifest, indent=2))
data.to_csv(RUN / "data.csv", index=False)
np.save(RUN / "labels.npy", y)

versions = {name: version(name) for name in [
    "torch", "transformers", "faster-whisper", "catboost", "scikit-learn", "numpy", "pandas"
]}
(RUN / "versions.json").write_text(json.dumps(versions, indent=2))
print("Run directory:", RUN)
display(pd.Series(versions, name="version").to_frame())


## 5. Optional reuse of the previous run

This step does nothing when `REUSE_RUN=None`. Reuse is explicit: records must match by **split and filename**,
and training labels must match. Only the expected transcript/acoustic fields are copied; old model-score columns
are never used as training features. Imported caches retain their original ASR decoding behavior.

Exact historical scores require the original data, transcripts, folds, model revisions and package versions.
The earlier run mixed beam sizes after an ASR failure, so a fresh transcription run may produce different scores.


In [ ]:
reuse_notes = []
old_config = None

if REUSE_RUN is not None:
    REUSE_RUN = Path(REUSE_RUN)
    old_data = pd.read_csv(REUSE_RUN / "data.csv")
    assert old_data[["split", "filename"]].equals(data[["split", "filename"]]), "Reuse record order differs."
    assert np.array_equal(np.load(REUSE_RUN / "labels.npy"), y), "Reuse training labels differ."
    old_config = json.loads((REUSE_RUN / "config.json").read_text())
    assert old_config.get("asr") == MODEL_IDS["asr"], "Cached transcripts used a different ASR model."
    (RUN / "reused_original_config.json").write_text(json.dumps(old_config, indent=2))

    if REUSE_LARGE_RUN is None:
        large_candidates = [p for p in REUSE_RUN.glob("wavlm_large_*") if (p / "config.json").exists()]
        assert len(large_candidates) <= 1, "Set REUSE_LARGE_RUN to the intended cached large-model run."
        REUSE_LARGE_RUN = large_candidates[0] if large_candidates else None

    for stage, source_dir, prefix, dimension in [
        ("asr", REUSE_RUN, "asr", None),
        ("audio_base", REUSE_RUN, "audio", 1536),
        ("audio_large", REUSE_LARGE_RUN, "audio", 2048),
    ]:
        if source_dir is None:
            continue
        source_dir = Path(source_dir)
        if stage != "asr":
            source_config = json.loads((source_dir / "config.json").read_text())
            assert source_config.get("audio") == MODEL_IDS[stage], f"Wrong cached encoder for {stage}"
            source_data = pd.read_csv(source_dir / "data.csv")
            assert source_data[["split", "filename"]].equals(data[["split", "filename"]]), f"Wrong records for {stage}"
            (RUN / f"reused_{stage}_config.json").write_text(json.dumps(source_config, indent=2))
        copied = 0
        for index in range(len(data)):
            extension = ".json" if stage == "asr" else ".npz"
            source = source_dir / f"{prefix}_{index}{extension}"
            destination = RUN / stage / f"{index}{extension}"
            if not source.exists() or destination.exists():
                continue
            if stage == "asr":
                record = json.loads(source.read_text())
                assert {"text", "duration", "logprob", "nospeech", "segments"}.issubset(record)
                assert isinstance(record["text"], str)
            else:
                with np.load(source) as record:
                    assert record["embedding"].shape == (dimension,)
                    assert record["numeric"].shape == (8,)
                    assert np.isfinite(record["embedding"]).all() and np.isfinite(record["numeric"]).all()
            shutil.copyfile(source, destination)
            copied += 1
        reuse_notes.append(f"{stage}: imported {copied} cached records from {source_dir}")

print("\n".join(reuse_notes) if reuse_notes else "Fresh run: no previous caches imported.")


## 6. GPU worker — shared utilities

The next five cells write a small Python helper included entirely in this notebook. Separate processes let
each GPU work independently and release model memory when a stage finishes. The actual modeling functions
are shown below as ordinary, commented Python. Run these cells in order; the first creates the file and the
following cells append functions.


In [ ]:
%%writefile /kaggle/working/shl_final_worker.py
import os
import sys
import json
import math
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import soundfile as sf
from scipy.signal import resample_poly

def save_arrays(path, **arrays):
    """Write atomically: an interruption cannot leave a partial cached result."""
    temporary = path.with_suffix(".tmp")
    with open(temporary, "wb") as handle:
        np.savez_compressed(handle, **arrays)
    temporary.replace(path)

def load_audio(path):
    signal, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    signal = signal.mean(axis=1)
    if not len(signal) or not np.isfinite(signal).all():
        raise ValueError(f"Empty or nonfinite audio: {path}")
    if sample_rate != 16000:
        divisor = math.gcd(sample_rate, 16000)
        signal = resample_poly(signal, 16000 // divisor, sample_rate // divisor).astype("float32")
    return np.pad(signal, (0, max(0, 400 - len(signal))))


### 6.1 Whisper transcription

Whisper produces text and confidence statistics; it does not assign the grammar score. English is fixed,
repetitions are retained, and no grammar-correction prompt is supplied. Word timestamps are disabled.
No word-timing pause features are included, ensuring the repaired and cached transcripts use a consistent feature schema.


In [ ]:
%%writefile -a /kaggle/working/shl_final_worker.py

def run_asr(run, cfg, data, jobs, device):
    from faster_whisper import WhisperModel

    model = WhisperModel(
        cfg["models"]["asr"], device=device,
        compute_type="int8_float16" if device == "cuda" else "int8",
        cpu_threads=torch.get_num_threads(), num_workers=1,
    )
    for number, index in enumerate(jobs):
        destination = run / "asr" / f"{index}.json"
        if destination.exists():
            continue
        segments, info = model.transcribe(
            data.path.iloc[index], language="en", beam_size=cfg["asr_beam_size"],
            temperature=0, condition_on_previous_text=False, vad_filter=False,
            word_timestamps=False, repetition_penalty=1.0, no_repeat_ngram_size=0,
        )
        segments = list(segments)
        record = {
            "text": " ".join(segment.text.strip() for segment in segments),
            "duration": float(info.duration),
            "logprob": float(np.mean([s.avg_logprob for s in segments])) if segments else -10.0,
            "nospeech": float(np.mean([s.no_speech_prob for s in segments])) if segments else 1.0,
            "segments": len(segments),
        }
        temporary = destination.with_suffix(".tmp")
        temporary.write_text(json.dumps(record))
        temporary.replace(destination)
        if number % 25 == 0:
            print(f"Transcribed {number + 1}/{len(jobs)} assigned files", flush=True)


### 6.2 WavLM audio representations

For both WavLM sizes, process the **whole recording** as 8-second chunks. Each output frame is a numeric
speech representation. Sum the frame values and squared values across chunks to compute a recording-level
mean and standard deviation. This uses little memory but loses the order between chunks.

The base model produces 1,536 pooled features and the large model produces 2,048. Eight additional features
describe duration, loudness quantiles, low-energy frames and zero crossings. Their definitions match the earlier runs.


In [ ]:
%%writefile -a /kaggle/working/shl_final_worker.py

def run_audio(run, cfg, data, jobs, stage, device):
    from transformers import AutoModel, Wav2Vec2FeatureExtractor

    model = AutoModel.from_pretrained(cfg["models"][stage], local_files_only=True).to(device).eval()
    extractor = Wav2Vec2FeatureExtractor.from_pretrained(cfg["models"][stage], local_files_only=True)
    chunk_samples = cfg["audio_chunk_seconds"] * 16000

    @torch.inference_mode()
    def chunk_moments(chunk, mixed_precision=True):
        chunk = np.pad(chunk, (0, max(0, 400 - len(chunk))))
        inputs = extractor(chunk, sampling_rate=16000, return_tensors="pt").input_values.to(device)
        with torch.autocast("cuda", dtype=torch.float16,
                            enabled=device == "cuda" and mixed_precision):
            hidden = model(inputs).last_hidden_state[0].float()
        if not torch.isfinite(hidden).all():
            raise FloatingPointError("Nonfinite audio representation")
        return hidden.sum(0).cpu().numpy(), hidden.square().sum(0).cpu().numpy(), len(hidden)

    for number, index in enumerate(jobs):
        destination = run / stage / f"{index}.npz"
        if destination.exists():
            continue
        signal = load_audio(data.path.iloc[index])
        total = squared_total = None
        frame_count = 0
        for start in range(0, len(signal), chunk_samples):
            chunk = signal[start:start + chunk_samples]
            try:
                part, squared_part, count = chunk_moments(chunk)
            except FloatingPointError:
                part, squared_part, count = chunk_moments(chunk, mixed_precision=False)
            total = part if total is None else total + part
            squared_total = squared_part if squared_total is None else squared_total + squared_part
            frame_count += count

        mean = total / frame_count
        std = np.sqrt(np.maximum(squared_total / frame_count - mean**2, 0))
        embedding = np.concatenate([mean, std]).astype("float32")

        # 20-ms frames describe loudness and the fraction of low-energy audio.
        framed = np.pad(signal, (0, (-len(signal)) % 320)).reshape(-1, 320)
        rms = np.sqrt(np.mean(framed**2, axis=1) + 1e-10)
        numeric = np.array([
            len(signal) / 16000,
            *np.quantile(rms, [0.1, 0.25, 0.5, 0.75, 0.9]),
            np.mean(rms < max(0.005, float(rms.max()) * 0.05)),
            np.mean(np.signbit(signal[1:]) != np.signbit(signal[:-1])),
        ], dtype="float32")
        assert np.isfinite(embedding).all() and np.isfinite(numeric).all()
        save_arrays(destination, embedding=embedding, numeric=numeric)
        if number % 50 == 0:
            print(f"{stage}: {number + 1}/{len(jobs)} assigned files", flush=True)


### 6.3 DeBERTa transcript regressor

DeBERTa learns a continuous score from each transcript. Its large word-embedding table is frozen to reduce
optimizer memory; Transformer layers and the regression head remain trainable. Four epochs and the learning
rate are fixed before evaluation. Validation labels are **not** used for checkpoint selection.

For each fold, we retain validation predictions, test predictions and separate in-sample predictions.
We do not retain large model/optimizer checkpoints.


In [ ]:
%%writefile -a /kaggle/working/shl_final_worker.py

def run_text(run, cfg, data, jobs, device):
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import get_cosine_schedule_with_warmup

    tokenizer = AutoTokenizer.from_pretrained(cfg["models"]["text"], use_fast=False, local_files_only=True)
    transcripts = [json.loads((run / "asr" / f"{i}.json").read_text())["text"] for i in range(len(data))]
    encoded = tokenizer(transcripts, truncation=True, max_length=cfg["text_max_length"])
    examples = [{key: values[i] for key, values in encoded.items()} for i in range(len(data))]
    labels = np.load(run / "labels.npy")
    folds = np.load(run / "folds.npy")
    n_train = len(labels)

    def collate(indices):
        batch = tokenizer.pad([examples[i] for i in indices], padding=True,
                              pad_to_multiple_of=8, return_tensors="pt")
        return batch, torch.tensor(indices, dtype=torch.long)

    def make_loader(indices, shuffle=False):
        return DataLoader(list(map(int, indices)), batch_size=cfg["text_batch_size"],
                          shuffle=shuffle, collate_fn=collate, num_workers=0)

    for fold in jobs:
        destination = run / "text" / f"{fold}.npz"
        if destination.exists():
            continue
        random.seed(cfg["seed"] + fold)
        np.random.seed(cfg["seed"] + fold)
        torch.manual_seed(cfg["seed"] + fold)

        model = AutoModelForSequenceClassification.from_pretrained(
            cfg["models"]["text"], num_labels=1, problem_type="regression", local_files_only=True,
        )
        model.deberta.embeddings.word_embeddings.weight.requires_grad_(False)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.to(device)
        optimizer = torch.optim.AdamW(
            (parameter for parameter in model.parameters() if parameter.requires_grad),
            lr=cfg["text_learning_rate"], weight_decay=0.01,
        )
        train_ids = np.flatnonzero(folds != fold)
        valid_ids = np.flatnonzero(folds == fold)
        loader = make_loader(train_ids, shuffle=True)
        accumulation = max(1, cfg["text_effective_batch"] // cfg["text_batch_size"])
        steps = math.ceil(len(loader) / accumulation) * cfg["text_epochs"]
        scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, int(0.1 * steps)), steps)
        scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

        for epoch in range(cfg["text_epochs"]):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            losses = []
            for batch_number, (batch, indices) in enumerate(loader):
                batch = {key: value.to(device) for key, value in batch.items()}
                target = torch.tensor(labels[indices.numpy()], device=device)
                group_size = min(accumulation, len(loader) - (batch_number // accumulation) * accumulation)
                with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
                    prediction = model(**batch).logits.flatten()
                    loss = torch.nn.functional.mse_loss(prediction.float(), target)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Nonfinite DeBERTa loss; inspect the worker log.")
                losses.append(loss.item())
                scaler.scale(loss / group_size).backward()
                if (batch_number + 1) % accumulation == 0 or batch_number + 1 == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    previous_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale() >= previous_scale:
                        scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            print(f"Fold {fold + 1}, epoch {epoch + 1}: batch MSE {np.mean(losses):.4f}", flush=True)

        @torch.inference_mode()
        def predict(indices):
            model.eval()
            predictions = []
            for batch, _ in make_loader(indices):
                with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
                    values = model(**{key: value.to(device) for key, value in batch.items()}).logits.flatten()
                predictions.extend(values.float().cpu().tolist())
            values = np.asarray(predictions, dtype="float32")
            assert np.isfinite(values).all()
            return values

        save_arrays(destination, valid_ids=valid_ids, valid=predict(valid_ids),
                    train_ids=train_ids, fitted=predict(train_ids), test=predict(range(n_train, len(data))))
        del model, optimizer, scheduler, scaler
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()


### 6.4 Worker entry point
Every worker receives a fixed slice of the pending jobs. This avoids duplicating or skipping jobs when
the other GPU finishes files concurrently. Already-completed outputs are skipped on resume.


In [ ]:
%%writefile -a /kaggle/working/shl_final_worker.py

if __name__ == "__main__":
    run = Path(sys.argv[1])
    stage = sys.argv[2]
    rank, worker_count = int(sys.argv[3]), int(sys.argv[4])
    device = sys.argv[5]
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", "1")))
    torch.set_num_interop_threads(1)
    cfg = json.loads((run / "config.json").read_text())
    data = pd.read_csv(run / "data.csv")
    jobs = json.loads((run / f"jobs_{stage}.json").read_text())[rank::worker_count]

    if stage == "asr":
        run_asr(run, cfg, data, jobs, device)
    elif stage in ("audio_base", "audio_large"):
        run_audio(run, cfg, data, jobs, stage, device)
    elif stage == "text":
        run_text(run, cfg, data, jobs, device)
    else:
        raise ValueError(f"Unknown stage: {stage}")


## 7. Run the GPU stages safely

The launcher bounds concurrent workers by available host RAM, divides CPU threads between them, and
shows one updating progress indicator. GPU resource failures retry unfinished work on CPU after the
GPU processes exit; this is slower and is recorded. Other errors stop with the actual log.
These measures reduce OOM/disk risk but cannot guarantee that every Kaggle environment will succeed.


In [ ]:
import site
import time

WORKER = OUT / "shl_final_worker.py"
worker_text = WORKER.read_text()
compile(worker_text, str(WORKER), "exec")
worker_hash = hashlib.sha256(worker_text.encode()).hexdigest()
signature_path = RUN / "worker_signature.txt"
if signature_path.exists():
    assert signature_path.read_text() == worker_hash, "Worker changed: increase pipeline_version in manifest before rerunning."
signature_path.write_text(worker_hash)

cuda_library_dirs = [str(path) for base in site.getsitepackages()
                     for path in Path(base).glob("nvidia/*/lib")]
fallback_events = []

def output_path(stage, index):
    extension = ".json" if stage == "asr" else ".npz"
    return RUN / stage / f"{index}{extension}"

def launch(stage):
    total = SETTINGS["folds"] if stage == "text" else len(data)
    pending = [i for i in range(total) if not output_path(stage, i).exists()]
    print(f"{stage}: {total - len(pending)}/{total} cached")
    if not pending:
        return
    require_free_disk(0.3)
    (RUN / f"jobs_{stage}.json").write_text(json.dumps(pending))
    ram_slots = max(1, int(psutil.virtual_memory().available / 2**30 // 4))
    workers = min(max(1, GPU_COUNT), len(pending), ram_slots)
    status = display(f"{stage}: starting {workers} worker(s)", display_id=True)
    processes = []

    try:
        for rank in range(workers):
            threads = max(1, CPU_CORES // workers + int(rank < CPU_CORES % workers))
            env = os.environ.copy()
            env.update(CUDA_VISIBLE_DEVICES=str(rank) if GPU_COUNT else "",
                       OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads),
                       OPENBLAS_NUM_THREADS=str(threads),
                       LD_LIBRARY_PATH=":".join(cuda_library_dirs + [env.get("LD_LIBRARY_PATH", "")]))
            args = [sys.executable, "-u", str(WORKER), str(RUN), stage,
                    str(rank), str(workers), "cuda" if GPU_COUNT else "cpu"]
            log = RUN / "logs" / f"{stage}_{rank}.log"
            handle = open(log, "w")
            process = subprocess.Popen(args, env=env, stdout=handle, stderr=subprocess.STDOUT)
            processes.append((process, handle, log, args, env))
        while any(process.poll() is None for process, *_ in processes):
            done = sum(output_path(stage, i).exists() for i in range(total))
            status.update(f"{stage}: {done}/{total} complete")
            time.sleep(5)
    finally:
        for process, handle, *_ in processes:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            handle.close()

    for process, _, log, args, env in processes:
        if process.returncode == 0:
            continue
        message = log.read_text(errors="replace")
        resource_error = process.returncode == -9 or any(
            term in message.lower() for term in ("out of memory", "cuda", "cudnn", "cublas")
        )
        if not GPU_COUNT or not resource_error:
            raise RuntimeError(f"{stage} exited with {process.returncode}:\n{message[-5000:]}")
        print(f"{stage}: retrying unfinished worker {args[-3]} jobs on CPU.")
        fallback_events.append({"stage": stage, "worker": args[-3], "reason": message[-1500:]})
        args[-1] = "cpu"
        env.update(CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS=str(CPU_CORES),
                   MKL_NUM_THREADS=str(CPU_CORES), OPENBLAS_NUM_THREADS=str(CPU_CORES))
        with open(log, "a") as handle:
            retry = subprocess.run(args, env=env, stdout=handle, stderr=subprocess.STDOUT)
        if retry.returncode:
            raise RuntimeError(log.read_text(errors="replace")[-5000:])

    assert all(output_path(stage, i).exists() for i in range(total))
    status.update(f"{stage}: {total}/{total} complete")
    (RUN / "fallbacks.json").write_text(json.dumps(fallback_events, indent=2))

launch("asr")
launch("audio_base")
launch("audio_large")


## 8. Build interpretable supporting features

Both audio encoders use identical auxiliary features. Word counts are recomputed from **all** transcripts,
including reused ones. The two timing-pause fields from the failed word-alignment experiment are excluded.
Whisper confidence and text statistics are only proxies; none directly measures grammatical correctness.

| Feature group | Examples |
|---|---|
| Acoustic statistics | Duration, loudness quantiles, low-energy fraction, zero-crossing rate |
| ASR statistics | Average log probability, no-speech probability, segment count, words/second |
| Transcript statistics | Word count, unique-word ratio, average word length, repeated words, filler fraction |


In [ ]:
import re

transcript_records = [json.loads((RUN / "asr" / f"{i}.json").read_text()) for i in range(len(data))]
for record in transcript_records:
    words = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", record["text"])
    record["word_count"] = len(words)
    record["words_per_second"] = len(words) / max(float(record["duration"]), 1.0)

asr = pd.DataFrame(transcript_records)
asr_columns = ["duration", "logprob", "nospeech", "segments", "word_count", "words_per_second"]
asr = asr[["text"] + asr_columns]
pd.concat([data[["filename", "split"]], asr], axis=1).to_csv(OUT / "transcripts.csv", index=False)

def transcript_statistics(text):
    words = re.findall(r"[a-z']+", text.lower())
    count = max(1, len(words))
    return [
        len(words), len(set(words)) / count, sum(map(len, words)) / count,
        sum(a == b for a, b in zip(words, words[1:])) / count,
        sum(word in {"uh", "um", "erm", "hmm"} for word in words) / count,
    ]

def load_acoustic_features(stage):
    embeddings, numeric = [], []
    for index in range(len(data)):
        with np.load(RUN / stage / f"{index}.npz") as record:
            embeddings.append(record["embedding"])
            numeric.append(record["numeric"])
    return np.asarray(embeddings, dtype=np.float32), np.asarray(numeric, dtype=np.float32)

base_embeddings, acoustic_statistics = load_acoustic_features("audio_base")
large_embeddings, _ = load_acoustic_features("audio_large")
auxiliary = np.column_stack([
    acoustic_statistics, asr[asr_columns].to_numpy(dtype=np.float32),
    np.asarray([transcript_statistics(text) for text in asr.text], dtype=np.float32),
]).astype(np.float32)
X_base = np.column_stack([base_embeddings, auxiliary]).astype(np.float32)
X_large = np.column_stack([large_embeddings, auxiliary]).astype(np.float32)
assert np.isfinite(X_base).all() and np.isfinite(X_large).all()
print("WavLM-base-plus features:", X_base.shape)
print("WavLM-large features:", X_large.shape)
display(pd.concat([train[["filename", "label"]].head(), asr[["text", "word_count"]].head()], axis=1))


## 9. Five-fold validation

Each recording is predicted once by models that **did not train on it**. These are out-of-fold (OOF)
predictions. Rare scores are merged into coarse bins for stratification. Exact matching nonempty normalized
transcripts are kept together in one fold; empty transcripts get separate groups.

No speaker IDs were supplied. Transcript grouping does not establish speaker-independent validation, and
does not rule out similar or duplicate audio. Test scores can therefore differ from OOF estimates.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

groups = []
for index, text in enumerate(asr.text.iloc[:n_train]):
    normalized = " ".join(re.findall(r"[a-z']+", text.lower()))
    groups.append(normalized if normalized else f"__empty_{index}")
bins = np.digitize(y, [1.75, 2.75, 3.75, 4.75])
folds = np.full(n_train, -1, dtype=int)
splitter = StratifiedGroupKFold(n_splits=SETTINGS["folds"], shuffle=True, random_state=SETTINGS["seed"])
for fold, (_, valid_ids) in enumerate(splitter.split(np.zeros(n_train), bins, groups)):
    folds[valid_ids] = fold
assert (folds >= 0).all()
np.save(RUN / "folds.npy", folds)
train.assign(fold=folds)[["filename", "label", "fold"]].to_csv(OUT / "fold_assignments.csv", index=False)
display(pd.crosstab(pd.Series(folds, name="fold"), pd.Series(y, name="label")))

# Optional reuse of DeBERTa predictions: require matching settings, fold indices and labels.
if REUSE_RUN is not None:
    matching_text_settings = all(old_config.get(old) == SETTINGS[new] for old, new in [
        ("seed", "seed"), ("folds", "folds"), ("epochs", "text_epochs"),
        ("max_length", "text_max_length"), ("batch", "text_batch_size"),
        ("effective_batch", "text_effective_batch"), ("lr", "text_learning_rate"),
    ]) and old_config.get("text") == MODEL_IDS["text"]
    all_asr_reused = all((Path(REUSE_RUN) / f"asr_{i}.json").exists() for i in range(len(data)))
    if matching_text_settings and all_asr_reused:
        for fold in range(SETTINGS["folds"]):
            source = Path(REUSE_RUN) / f"fold_{fold}.npz"
            destination = RUN / "text" / f"{fold}.npz"
            if not source.exists() or destination.exists():
                continue
            with np.load(source) as record:
                matching_ids = np.array_equal(record["valid_ids"], np.flatnonzero(folds == fold))
                matching_ids &= np.array_equal(record["train_ids"], np.flatnonzero(folds != fold))
                valid_predictions = all(np.isfinite(record[key]).all() for key in ("valid", "test", "fitted"))
                if matching_ids and valid_predictions and len(record["test"]) == n_test:
                    shutil.copyfile(source, destination)
                    reuse_notes.append(f"Imported DeBERTa fold {fold}")
launch("text")


## 10. Approach A — WavLM-base-plus + CatBoost

CatBoost learns a nonlinear mapping from the pooled speech representation and supporting statistics to a
continuous score. It uses 900 trees, depth 5, learning rate 0.035 and L2 regularization 10, matching the earlier
experiment. It is trained on CPU to avoid competing with speech models for GPU memory.

The helper below also trains Approach C with identical folds. CatBoost uses every available CPU core;
SVR runs on CPU. Models are fitted fold by fold to bound RAM, and completed predictions are cached.


In [ ]:
from catboost import CatBoostRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from threadpoolctl import threadpool_limits

CATBOOST_PARAMETERS = dict(iterations=900, depth=5, learning_rate=0.035, l2_leaf_reg=10,
                           loss_function="RMSE", allow_writing_files=False, verbose=False)
SVR_PARAMETERS = dict(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale", cache_size=256)

def fit_audio_regressors(features, experiment_name, include_svr=False):
    model_names = ["catboost"] if not include_svr else ["svr", "catboost"]
    columns = len(model_names)
    oof = np.full((n_train, columns), np.nan)
    test_predictions = np.zeros((n_test, columns))
    fitted_sum = np.zeros((n_train, columns))
    fitted_count = np.zeros(n_train)

    signature = hashlib.sha256(features.tobytes() + y.tobytes() + folds.tobytes()
                               + json.dumps([CATBOOST_PARAMETERS, SVR_PARAMETERS, versions], sort_keys=True).encode()).hexdigest()[:12]
    cache_dir = RUN / "regressors" / f"{experiment_name}_{signature}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    for fold in range(SETTINGS["folds"]):
        train_ids = np.flatnonzero(folds != fold)
        valid_ids = np.flatnonzero(folds == fold)
        cache_path = cache_dir / f"{fold}.npz"
        if not cache_path.exists():
            models = []
            if include_svr:
                models.append(make_pipeline(StandardScaler(), SVR(**SVR_PARAMETERS)))
            models.append(CatBoostRegressor(**CATBOOST_PARAMETERS,
                          random_seed=SETTINGS["seed"] + fold, thread_count=CPU_CORES))
            with threadpool_limits(limits=CPU_CORES):
                for model in models:
                    model.fit(features[train_ids], y[train_ids])
                valid = np.column_stack([model.predict(features[valid_ids]) for model in models])
                test_values = np.column_stack([model.predict(features[n_train:]) for model in models])
                fitted = np.column_stack([model.predict(features[train_ids]) for model in models])
            temporary = cache_path.with_suffix(".tmp")
            with open(temporary, "wb") as handle:
                np.savez_compressed(handle, valid_ids=valid_ids, train_ids=train_ids,
                                    valid=valid, test=test_values, fitted=fitted)
            temporary.replace(cache_path)
            del models

        with np.load(cache_path) as record:
            assert np.array_equal(record["valid_ids"], valid_ids)
            assert np.array_equal(record["train_ids"], train_ids)
            oof[valid_ids] = record["valid"]
            test_predictions += record["test"] / SETTINGS["folds"]
            fitted_sum[train_ids] += record["fitted"]
        fitted_count[train_ids] += 1
        print(f"{experiment_name}: fold {fold + 1}/{SETTINGS['folds']} complete")

    assert np.isfinite(oof).all() and np.isfinite(test_predictions).all()
    assert (fitted_count > 0).all()
    return {"names": model_names, "oof": np.clip(oof, 0, 5),
            "test": np.clip(test_predictions, 0, 5),
            "fitted": np.clip(fitted_sum / fitted_count[:, None], 0, 5)}

base_result = fit_audio_regressors(X_base, "wavlm_base_catboost")


## 11. Approach B — DeBERTa + audio ensemble, without TF-IDF

The previous ensemble assigned weights 0.65 to DeBERTa, 0.25 to audio CatBoost, and 0.10 to TF-IDF Ridge.
Here TF-IDF is removed completely. Preserve the remaining ratio by normalizing **0.65:0.25**, giving
**72.22% DeBERTa and 27.78% audio CatBoost**. These are fixed weights, not a validation-optimized choice.

This is a **new two-model ensemble**. The previous three-model leaderboard score of 0.4320 does not belong
to this version. Its actual metrics are computed below.


In [ ]:
text_oof = np.full(n_train, np.nan)
text_test = np.zeros(n_test)
text_fitted_sum = np.zeros(n_train)
text_fitted_count = np.zeros(n_train)

for fold in range(SETTINGS["folds"]):
    with np.load(RUN / "text" / f"{fold}.npz") as record:
        valid_ids, train_ids = record["valid_ids"], record["train_ids"]
        assert np.array_equal(valid_ids, np.flatnonzero(folds == fold))
        assert np.array_equal(train_ids, np.flatnonzero(folds != fold))
        text_oof[valid_ids] = record["valid"]
        text_test += record["test"] / SETTINGS["folds"]
        text_fitted_sum[train_ids] += record["fitted"]
        text_fitted_count[train_ids] += 1

assert np.isfinite(text_oof).all() and np.isfinite(text_test).all()
text_result = {"oof": np.clip(text_oof, 0, 5), "test": np.clip(text_test, 0, 5),
               "fitted": np.clip(text_fitted_sum / text_fitted_count, 0, 5)}
text_weight, audio_weight = 0.65 / 0.90, 0.25 / 0.90
ensemble_result = {
    key: text_weight * text_result[key] + audio_weight * base_result[key][:, 0]
    for key in ("oof", "test", "fitted")
}
print(f"Weights: DeBERTa {text_weight:.4f}; audio CatBoost {audio_weight:.4f}")


## 12. Approach C — WavLM-large + SVR/CatBoost

Upgrade the speech representation while keeping the supporting features and folds unchanged.
SVR learns a smooth nonlinear mapping using an RBF kernel; StandardScaler is fitted **inside each fold**.
CatBoost supplies a different regression model. Average their clipped predictions with fixed **50:50 weights**.
The large encoder remains frozen, so this experiment does not fine-tune hundreds of millions of parameters.


In [ ]:
large_result = fit_audio_regressors(X_large, "wavlm_large_svr_catboost", include_svr=True)
large_blend = {key: large_result[key].mean(axis=1) for key in ("oof", "test", "fitted")}

approaches = {
    "A — Audio + CatBoost": {key: base_result[key][:, 0] for key in ("oof", "test", "fitted")},
    "B — Text/audio, no TF-IDF": ensemble_result,
    "C — WavLM-large blend": large_blend,
}
components = {
    "DeBERTa": text_result,
    "WavLM-large SVR": {key: large_result[key][:, 0] for key in ("oof", "test", "fitted")},
    "WavLM-large CatBoost": {key: large_result[key][:, 1] for key in ("oof", "test", "fitted")},
}


## 13. Required training RMSE and validation results

- **OOF RMSE:** error on training recordings predicted by models that did not train on them; lower is better.
- **OOF Pearson:** correlation between actual and predicted scores; higher is better.
- **In-sample training RMSE:** uses predictions from the fold models that did train on each recording.
  This satisfies the training-metric requirement but is optimistic; it must not be reported as unseen-data performance.

The test predictions average all five fold models. No test labels are inspected. The table below contains
only metrics calculated by this execution; historical leaderboard results are discussed separately.


In [ ]:
def calculate_metrics(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    correlation = float(np.corrcoef(actual, predicted)[0, 1]) if np.std(actual) > 0 and np.std(predicted) > 0 else np.nan
    return {"RMSE": float(np.sqrt(np.mean((actual - predicted)**2))), "Pearson": correlation}

def metrics_row(name, result):
    validation = calculate_metrics(y, result["oof"])
    training = calculate_metrics(y, result["fitted"])
    return {"Model": name, "OOF_RMSE": validation["RMSE"], "OOF_Pearson": validation["Pearson"],
            "Training_RMSE_in_sample": training["RMSE"]}

metrics_table = pd.DataFrame([metrics_row(name, result) for name, result in approaches.items()])
component_table = pd.DataFrame([metrics_row(name, result) for name, result in components.items()])
display(metrics_table.round(4))
display(component_table.round(4))

mean_baseline = np.array([y[folds != fold].mean() for fold in folds])
print("Fold-training-mean baseline:", calculate_metrics(y, mean_baseline))
print(f"REQUIRED TRAINING RMSE — final Approach C: {calculate_metrics(y, large_blend['fitted'])['RMSE']:.4f}")
print(f"OOF RMSE — final Approach C: {calculate_metrics(y, large_blend['oof'])['RMSE']:.4f}")
metrics_table.to_csv(OUT / "metrics.csv", index=False)
component_table.to_csv(OUT / "component_metrics.csv", index=False)

oof_export = train[["filename", "label"]].copy()
oof_export["fold"] = folds
for key, result in zip(("audio_catboost", "text_audio", "wavlm_large"), approaches.values()):
    oof_export[key] = result["oof"]
oof_export.to_csv(OUT / "oof_predictions.csv", index=False)


## 14. Visual comparison and error analysis

Compare overall errors, then inspect the final model's predictions, residuals and errors at each true score.
Good performance at zero does not establish good grammar discrimination among nonzero recordings, so both
groups are reported separately. A per-label result based on one or three examples is not a stable estimate.


In [ ]:
short_names = ["Audio + CatBoost", "Text/audio", "WavLM-large"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(short_names, metrics_table.OOF_RMSE, color=["#64748b", "#0d9488", "#2563eb"])
axes[0].set(title="Validation comparison", ylabel="OOF RMSE — lower is better")
axes[0].tick_params(axis="x", rotation=15)
axes[1].scatter(y, large_blend["oof"], s=18, alpha=0.4)
axes[1].plot([0, 5], [0, 5], "k--")
axes[1].set(title="WavLM-large: OOF predictions", xlabel="Actual score", ylabel="Predicted score")
axes[2].hist(large_blend["oof"] - y, bins=25, edgecolor="white", color="#2563eb")
axes[2].axvline(0, color="black", linestyle="--")
axes[2].set(title="WavLM-large: residuals", xlabel="Predicted − actual", ylabel="Count")
plt.tight_layout()
plt.savefig(OUT / "model_comparison.png", dpi=150)
plt.show()

label_rows = []
for label in np.unique(y):
    mask = y == label
    row = {"label": float(label), "count": int(mask.sum())}
    for short, result in zip(short_names, approaches.values()):
        row[short] = calculate_metrics(y[mask], result["oof"][mask])["RMSE"]
    row["large_mean_prediction"] = float(large_blend["oof"][mask].mean())
    label_rows.append(row)
per_label = pd.DataFrame(label_rows)
display(per_label.round(4))
per_label.to_csv(OUT / "errors_by_label.csv", index=False)

ax = per_label.set_index("label")[short_names].plot(marker="o", figsize=(9, 4))
ax.set(title="OOF error by true grammar score", xlabel="True score", ylabel="RMSE")
plt.tight_layout()
plt.savefig(OUT / "errors_by_label.png", dpi=150)
plt.show()

subgroup_rows = []
for subgroup, mask in [("Score zero", y == 0), ("Nonzero scores", y > 0)]:
    if mask.any():
        for name, result in approaches.items():
            subgroup_rows.append({"Group": subgroup, "Count": int(mask.sum()), "Model": name,
                                  **calculate_metrics(y[mask], result["oof"][mask])})
subgroup_metrics = pd.DataFrame(subgroup_rows)
display(subgroup_metrics.round(4))
subgroup_metrics.to_csv(OUT / "subgroup_metrics.csv", index=False)


## 15. Submission audit and exports

The supplied development logs showed a **204-row sample submission versus 216 test recordings**, with
some sample IDs absent from test.csv. Never assign predictions by row position to an inconsistent template.

- If the sample is a valid subset of test IDs, use its IDs and order.
- If sample IDs are missing from test.csv, export the actual test IDs and clearly report the mismatch.
- Do not fabricate scores for unknown template IDs.

The author's earlier accepted submissions used all 216 test IDs. The current dataset is still audited here.
Approach C is chosen as the final submission based on the prior experiment; this code does not tune against
the public leaderboard or automatically select the smallest OOF error.


In [ ]:
if sample is not None and set(sample.columns) == {"filename", "label"} and sample.filename.is_unique and len(sample):
    unknown_ids = set(sample.filename) - set(test.filename)
    if not unknown_ids:
        submission_ids = sample[["filename"]].copy()
        submission_note = f"Used sample IDs and order: {len(sample)} of {n_test} test rows."
    else:
        submission_ids = test[["filename"]].copy()
        submission_note = (f"Template mismatch: {len(unknown_ids)} sample IDs are absent from test.csv. "
                           f"Exported all {n_test} actual test IDs; confirm the required IDs with the competition.")
else:
    submission_ids = test[["filename"]].copy()
    submission_note = f"No usable two-column template; exported all {n_test} test IDs."

def export_submission(values, filename):
    lookup = test[["filename"]].copy()
    lookup["label"] = np.clip(values, 0, 5)
    result = submission_ids.merge(lookup, on="filename", how="left", sort=False, validate="one_to_one")
    assert result.filename.tolist() == submission_ids.filename.tolist()
    assert result.filename.is_unique and np.isfinite(result.label).all()
    assert result.label.between(0, 5).all()
    result.to_csv(OUT / filename, index=False)
    return result

submission_a = export_submission(approaches["A — Audio + CatBoost"]["test"], "submission_audio_catboost.csv")
submission_b = export_submission(ensemble_result["test"], "submission_text_audio.csv")
submission_c = export_submission(large_blend["test"], "submission_wavlm_large.csv")
submission_c.to_csv(OUT / "submission.csv", index=False)

all_test = test[["filename"]].copy()
for key, result in zip(("audio_catboost", "text_audio", "wavlm_large"), approaches.values()):
    all_test[key] = result["test"]
all_test.to_csv(OUT / "test_predictions_all.csv", index=False)
print(submission_note)
print("Final file:", OUT / "submission.csv")
display(submission_c.head())


## 16. Report for final evaluation

The following report is populated with this run's actual scores. Save the executed notebook so the report
and its plots are visible. Historical public leaderboard results below are **author-reported context**, not
scores generated or verified by this notebook:

| Historical experiment | Public leaderboard RMSE |
|---|---:|
| WavLM-base-plus + CatBoost with supporting features | 0.4473 |
| WavLM-large + 50:50 SVR/CatBoost with supporting features | 0.4135 |

The removed TF-IDF branch means Approach B differs from the earlier three-model ensemble. None of these
historical leaderboard values is a substitute for the evaluation outputs of the current execution.


In [ ]:
def markdown_table(frame):
    display_frame = frame.copy()
    for column in display_frame.select_dtypes(include="number"):
        display_frame[column] = display_frame[column].map(lambda value: f"{value:.4f}")
    header = "| " + " | ".join(map(str, display_frame.columns)) + " |"
    divider = "| " + " | ".join(["---"] * len(display_frame.columns)) + " |"
    rows = ["| " + " | ".join(map(str, row)) + " |" for row in display_frame.itertuples(index=False, name=None)]
    return "\n".join([header, divider] + rows)

final_validation = calculate_metrics(y, large_blend["oof"])
final_training = calculate_metrics(y, large_blend["fitted"])
report = f"""# Spoken Grammar Scoring — Evaluation Report
**Author: Priyansh Rathore**

## Task and data
Predict a continuous grammar score in [0,5] from speech. This run used {n_train} labeled recordings
and {n_test} unlabeled recordings. Test placeholder labels were never used. Repeated filenames were
resolved within their train/test folders, and identical resolved paths across splits were rejected.

## Preprocessing and representations
Whisper large-v3 generated English transcripts without a correction prompt. Word timestamps were
disabled for fresh transcription; fresh decoding used beam size {SETTINGS['asr_beam_size']}.
Waveforms were mixed to mono and resampled to 16 kHz. Frozen WavLM-base-plus and WavLM-large
processed complete recordings in {SETTINGS['audio_chunk_seconds']}-second chunks. Mean/std pooling
over output frames produced compact recording-level representations. Loudness, duration, ASR confidence
and simple transcript statistics supplemented the embeddings. No word-alignment pause features were used.

## Models
A: WavLM-base-plus features and supporting statistics → CatBoost.
B: DeBERTa transcript regression and A combined at {text_weight:.4f}/{audio_weight:.4f}; no TF-IDF.
C: WavLM-large features and the same supporting statistics → equal-weight SVR/CatBoost.
SVR uses fold-local standardization and an RBF kernel. CatBoost uses fixed hyperparameters.
DeBERTa uses four fixed epochs, gradient checkpointing and a frozen word-embedding table.

## Validation and results
Five stratified group folds keep exact nonempty normalized transcripts together. Each OOF prediction
comes from models that did not train on its recording. Validation labels were not used for early stopping
or blend-weight fitting. Test predictions average the five fold models.

{markdown_table(metrics_table)}

**Required in-sample training RMSE for the final model: {final_training['RMSE']:.4f}.**
**Final-model OOF RMSE: {final_validation['RMSE']:.4f}; OOF Pearson: {final_validation['Pearson']:.4f}.**
The in-sample metric averages models trained on each row and is optimistic. OOF metrics are more useful
for generalization assessment. The plots and per-label tables reveal score-dependent errors and bias.

## Final submission
Approach C is exported as submission.csv, with {len(submission_c)} rows. Separate files contain A and B.
{submission_note}

## Resources and reproducibility
Detected GPUs: {GPU_COUNT}; available CPU cores: {CPU_CORES}. Models ran in separate processes,
using bounded concurrency, short audio chunks and mixed precision. No optimizer/epoch checkpoints were saved.
Resource-related CPU fallback events recorded in this execution: {len(fallback_events)}.
Model revisions, package versions, cached predictions and fold assignments are recorded with the run.
Imported-cache notes: {'; '.join(reuse_notes) if reuse_notes else 'none; features were generated by this run'}.

## Limitations
ASR can change or misrecognize grammar. Audio features may exploit fluency, accent or recording quality,
which are not equivalent to grammar. Pooling loses order across chunks. Speaker IDs were unavailable;
the folds do not prove unseen-speaker performance. Rare labels have uncertain per-label estimates.
The experiment family was developed iteratively, so this is not an untouched final holdout. Public
leaderboard scores reflect a different subset and need not track OOF model rankings.
Exact historical results cannot be promised after fresh transcription or dependency/hardware changes.
The author previously reported 0.4135 public RMSE for C; that historical result is not computed by this notebook.
"""
(OUT / "final_report.md").write_text(report)
(RUN / "reuse_notes.json").write_text(json.dumps(reuse_notes, indent=2))
(RUN / "fallbacks.json").write_text(json.dumps(fallback_events, indent=2))
display(Markdown(report))
print("Finished. Save & Run All in Kaggle to retain an executed notebook for final evaluation.")


## References and deliverables

- [Whisper implementation and decoding options](https://github.com/SYSTRAN/faster-whisper)
- [WavLM-base-plus](https://huggingface.co/microsoft/wavlm-base-plus)
- [WavLM-large](https://huggingface.co/microsoft/wavlm-large)
- [DeBERTa-v3-base](https://huggingface.co/microsoft/deberta-v3-base)
- [CatBoost regression](https://catboost.ai/en/docs/concepts/python-reference_catboostregressor)
- [SVR and feature scaling](https://scikit-learn.org/stable/modules/svm.html)

**Main deliverables:** `submission.csv`, three named approach submissions, `metrics.csv`,
`component_metrics.csv`, `oof_predictions.csv`, `test_predictions_all.csv`, `errors_by_label.csv`,
`subgroup_metrics.csv`, `transcripts.csv`, `fold_assignments.csv`, three figures and `final_report.md`.

**Execution status of this distributed notebook:** notebook/worker syntax and local synthetic-data checks
were performed during preparation. The competition WAV files and Kaggle GPUs were not available in that
environment. Metrics and model-generated outputs remain empty until the notebook is executed on Kaggle;
the preparation checks do not establish end-to-end GPU success or a guaranteed competition score.
